Notebook para generar embeddings
Previamente ya se realizaron para pocos archivos usando un modelos de OpenAI y validando con Qdrant [rag_qdrant](https://github.com/Halsey26/embedding_PerAI/blob/main/rag_qdrant.ipynb)
Sin embargo, ahora son más de 30 archivos pdf, algunos incluso con 300 páginas. Por ende se plantea usar langchain para:
- Chunkenizado
- Embedding
- Almacenamiento - Qdrant
- Función búsqueda
Después se modularizará para detectar los pdfs y obtener los embeddings

Librerias para descargar
- %pip install -qU pypdf
- pip install langchain
- pip install langchain-community
- pip install sentence-transformers
- pip intall tiktoken

## fsdf
Detecta si un pdf ya ha sido procesado. Si en caso no ha sido procesado, se aplica las funciones y se marca como **hecho**.

In [ ]:
import os
import hashlib
from langchain_community.document_loaders import PyPDFLoader


In [61]:
import os
def archivo_contenido(archivo):
    if not os.path.exists(archivo): # si no existe el archivo lo crea
        with open(archivo, 'w') as file:
            pass

    # verifica su contenido
    with open(archivo, 'r') as file:
        docs_procesados= list(file.read().splitlines())
    
    # print(f'Documentos procesados: {docs_procesados}')
    return docs_procesados

In [40]:
procesados = archivo_contenido('procesados.txt')

Documentos procesados: ['601459542-High-Growth-Handbook-PDFDrive-en-Espanol.pdf']


In [58]:
procesados = archivo_contenido('procesados.txt')

Documentos procesados: ['223221647-ECN-BusinessPath-fulldoc.pdf']


In [66]:
ruta_docs_pdf= '../doc_pdf'
# carpeta_embeddings = ''

def generate_no_procesados(ruta_docs_pdf):
    procesados = archivo_contenido('procesados.txt')
    docs_no_procesados= []
    # verificamos los archivos en carpeta de docs
    for filename in os.listdir(ruta_docs_pdf):
        # verificar si el archivo se encuentra en procesados.txt
        if filename not in procesados:
            # print('El archivo no ha sido procesado')
            ruta_completa= os.path.join(ruta_docs_pdf,filename)
            docs_no_procesados.append(ruta_completa)
        # else:
        #     print('Todos los archivos han sido procesados')

    print(f'Documentos para procesar:\n  {docs_no_procesados}')
    return docs_no_procesados

docs_no_procesados= generate_no_procesados(ruta_docs_pdf)

Documentos para procesar:
  ['../doc_pdf/601459542-High-Growth-Handbook-PDFDrive-en-Espanol.pdf']


## Empieza el procesamiento

Extracción del texto 

In [16]:
from langchain_community.document_loaders import PyPDFLoader

def extraccion_page(ruta):
    loader = PyPDFLoader(ruta)
    pages = loader.load()
    # async for page in loader.alazy_load():
    #     pages.append(page)
    print('✅ Extracción realizada')
    return pages


Limpieza del texto

In [33]:
import re

def clean_text(text: str) -> str:
    text = re.sub(r'©.*?\n', '', text)  # remueve símbolos de copyright y similares
    text = re.sub(r'\n+', ' ', text)  # convierte múltiples saltos de línea en espacio
    text = re.sub(r'\s{2,}', ' ', text)  # remueve espacios extra
    return text.strip()


Creacción de la metadata, estructura planteada:
- documento_id
- nombre documento
- numero pagina
- total_pages

In [54]:
from pathlib import Path
import hashlib

def generate_metadata(ruta_completa, pages):
    filename = Path(ruta_completa).name
    document_id = hashlib.md5(filename.encode()).hexdigest() # codificamos solo el nombre del archivo
    total_pages = pages[0].metadata['total_pages']
    docs_metadata = []
    for page in pages:
        page_number = page.metadata['page_label'] # númeración correcta de la página
        
        metadata = {
            "document_id": document_id,
            "filename": filename,
            "page_number": page_number,
            "total_pages": total_pages,
        }
        page.page_content = clean_text(page.page_content) # cleaned_text = clean_text(page.page_content)
        
        docs_metadata.append(
            {
                'text': page.page_content, #cleaned_text, 
                'metadata': metadata
            }
        )
    print('✅ Generación Documentos con Metadata (Limpieza por página)')
    return docs_metadata

Generación de embeddings

In [35]:
from dotenv import load_dotenv
import os
from openai import OpenAI

load_dotenv()
api_key=os.getenv('OPENAI_API_KEY')
# api_key
cliente= OpenAI()
cliente

In [56]:
import tiktoken # estimar la cantidad de token
import time

def costo_tokens(tokens):
    costo = tokens*0.02 /10**6 # 1 millon de tokens equivale a 0.02 dólares

    return f'   Tokens: {tokens}\n   Costo Tokens: ${costo:.4f}'


def generate_embedd(docs_metadata):
    print(f'   Generando embedding: ...')
    modelo_openai = "text-embedding-3-small"
    encoding= tiktoken.encoding_for_model(modelo_openai)

    docs_embedd = []
    total_tokens= 0


    for doc in docs_metadata:
        texto= doc['text']

        # Generación de número de tokens
        tokens= encoding.encode(texto)
        nro_tokens = len(tokens)
        total_tokens += nro_tokens

        doc['metadata']['token']=nro_tokens # añado los tokens a la metadata
    
        start= time.time()
        #  Generación de embeddings
        response = cliente.embeddings.create(
            input= texto, 
            model = modelo_openai
        )
        finish= time.time()
        embedding= response.data[0].embedding
        
        # embedding= modelo_seleccionado.encode(doc['text'], normalize_embeddings= True)
        docs_embedd.append({
            'vector': embedding,  #con openai, directamente el embedding
            'text': texto, 
            'metadata': doc['metadata'] 

        })
    print(costo_tokens(total_tokens))
    print(f'   Tiempo del embedding: {finish-start:.4f} segundos')
    print('✅ Generación de Embeddings')
    return docs_embedd

# comprobar con lo que sale en playground

Exportación embedding

In [48]:
ruta_completa

'../doc_pdf/223221647-ECN-BusinessPath-fulldoc.pdf'

In [49]:
filename = Path(ruta_completa).name

import json
def exportacion_json(docs_embeding,filename):
    with open(f"../json_embedding/{filename}.json", "w", encoding="utf-8") as file:
        json.dump(docs_embeding, file, ensure_ascii=False, indent=2)

    # luego que finalice todo el proceso, hay que realizar una función para agregar el archivo a procesados.txt
    with open('procesados.txt', 'w') as file:
        file.write(filename+"\n")

    print('✅ Exportacción realizada')

# exportacion_json(filename)

Función completa 

In [57]:
docs_no_procesados

['../doc_pdf/223221647-ECN-BusinessPath-fulldoc.pdf']

In [73]:
import tqdm
prueba = ['../doc_pdf/223221647-ECN-BusinessPath-fulldoc.pdf']

# for ruta_archivo in  tqdm.tqdm(prueba):#docs_no_procesados:
def proceso_completo(docs_no_procesados):
    '''
    Parámetro de entrada: Lista con todas las rutas de los archivos no procesados
    '''
    for ruta_archivo in  docs_no_procesados:#prueba:
        filename = Path(ruta_archivo).name
        print(f'📌 Generando Embeddings para {filename} ...') # CAMBIO AQUI
        # definir una funcion para aplicar  
        time1= time.time()
        pags=extraccion_page(ruta_archivo)
        docs_metadata = generate_metadata(ruta_archivo, pags)
        docs_embedd= generate_embedd(docs_metadata)
        exportacion_json(docs_embedd,filename)
        time3=time.time()
        segundos= time3-time1 
        print(f'\nTiempo total: {segundos:.2f} segundos - {segundos/60:.2f} minutos')
        print('🎉 Realizado: Embeddings Generados Correctamente.\n\n')

# Funcion completa
# def procesamiento():
    # extracion texto
    # limpieza por pagina
    # creacion de docs_metadata
    # obtencion de embedding
    # exportación de embedding
    

In [55]:
proceso_completo(docs_no_procesados)

📌 Generando Embeddings para 223221647-ECN-BusinessPath-fulldoc.pdf ...
✅ Extracción realizada
✅ Generación Documentos con Metadata (Limpieza por página)
   Generando embedding: ...
   Tokens: 1938
   Costo Tokens: $0.0000
Tiempo del embedding: 0.5894 segundos
✅ Generación de Embeddings
✅ Exportacción realizada

Tiempo total: 3.92 segundos
🎉 Realizado: Embeddings Generados Correctamente.



Ya ahora que tengo el embedding demo vamos a modularizar

In [70]:
# actualizamos para docs_no_procesados
docs_no_procesados= generate_no_procesados(ruta_docs_pdf)

Documentos para procesar:
  ['../doc_pdf/601459542-High-Growth-Handbook-PDFDrive-en-Espanol.pdf']


In [71]:
proceso_completo(docs_no_procesados)

📌 Generando Embeddings para 601459542-High-Growth-Handbook-PDFDrive-en-Espanol.pdf ...
✅ Extracción realizada
✅ Generación Documentos con Metadata (Limpieza por página)
   Generando embedding: ...
   Tokens: 190165
   Costo Tokens: $0.0038
   Tiempo del embedding: 0.2444 segundos
✅ Generación de Embeddings
✅ Exportacción realizada

Tiempo total: 184.87 segundos
🎉 Realizado: Embeddings Generados Correctamente.




In [72]:
184.87/60

3.081166666666667

In [3]:
import langchain
import langchain_community